In [ ]:
# Step 1 - Data loading
import glob
import numpy as np
import pandas as pd

INPUT_ROOT = "/kaggle/input"
OUTPUT_DIR = "/kaggle/working"

GENDERS = ["M", "W"]
PREDICT_SEASON = 2026       # season we must forecast; its tournament results are never used for training
CUTOFF_DAY = 132            # ASSUMPTION: regular-season info stops here (checked in Step 2)
FIRST_VAL_SEASON = 2015     # first validation season
EXP_YEARS = 3               # how many past tournaments count as "programme experience"


def read(name):
    hits = glob.glob(f"{INPUT_ROOT}/**/{name}.csv", recursive=True)
    if not hits:
        raise FileNotFoundError(f"{name}.csv not found under {INPUT_ROOT}")
    return pd.read_csv(hits[0])


raw = {}
for g in GENDERS:
    raw[g] = {
        "teams": read(f"{g}Teams"),
        "seeds": read(f"{g}NCAATourneySeeds"),
        "reg": read(f"{g}RegularSeasonCompactResults"),
        "tourney": read(f"{g}NCAATourneyCompactResults"),
    }
sample_sub = read("SampleSubmissionStage2")

for g in GENDERS:
    for name, df in raw[g].items():
        print(f"{g} {name:8s} {df.shape[0]:>8,} rows x {df.shape[1]} cols")
print("sample submission:", sample_sub.shape)


In [ ]:
# Step 2 - Data cleaning
def clean_seeds(seeds):
    df = seeds.copy()
    df["SeedNum"] = df["Seed"].str[1:3].astype(int)
    return df[["Season", "TeamID", "SeedNum"]]


def clean_results(df, teams, label):
    df = df.drop_duplicates().copy()
    valid = set(teams["TeamID"])
    bad = ~df["WTeamID"].isin(valid) | ~df["LTeamID"].isin(valid)
    if bad.any():
        print(f"[{label}] dropping {bad.sum()} rows with unknown TeamID")
    df = df[~bad]
    assert df["WLoc"].isin(["H", "A", "N"]).all(), f"[{label}] unexpected WLoc value"
    return df


clean = {}
for g in GENDERS:
    r = raw[g]
    reg = clean_results(r["reg"], r["teams"], f"{g} reg")
    tour = clean_results(r["tourney"], r["teams"], f"{g} tourney")
    print(f"\n[{g}] regular season DayNum {reg['DayNum'].min()}-{reg['DayNum'].max()}, "
          f"tournament DayNum {tour['DayNum'].min()}-{tour['DayNum'].max()}")
    print(f"[{g}] tournament seasons {tour['Season'].min()}-{tour['Season'].max()}")
    n_before = len(reg)
    reg = reg[reg["DayNum"] <= CUTOFF_DAY]
    print(f"[{g}] rows removed by CUTOFF_DAY: {n_before - len(reg)}")
    clean[g] = {"seeds": clean_seeds(r["seeds"]), "reg": reg, "tourney": tour}


In [ ]:
# Step 3 - Feature Engineering

def season_win_pct(reg):
    wins = reg.groupby(["Season", "WTeamID"]).size().rename_axis(["Season", "TeamID"]).rename("Wins")
    losses = reg.groupby(["Season", "LTeamID"]).size().rename_axis(["Season", "TeamID"]).rename("Losses")
    t = pd.concat([wins, losses], axis=1).fillna(0).reset_index()
    t["WinPct"] = t["Wins"] / (t["Wins"] + t["Losses"])
    return t[["Season", "TeamID", "WinPct"]]


def away_neutral_win_pct(reg):
    """Win% using only games NOT played at the team's own home court (tournament is neutral-court)."""
    w = pd.DataFrame({"Season": reg["Season"], "TeamID": reg["WTeamID"], "Win": 1.0,
                      "Home": reg["WLoc"] == "H"})          # winner was at home
    l = pd.DataFrame({"Season": reg["Season"], "TeamID": reg["LTeamID"], "Win": 0.0,
                      "Home": reg["WLoc"] == "A"})          # loser was at home when winner was away
    both = pd.concat([w, l])
    both = both[~both["Home"]]
    return (both.groupby(["Season", "TeamID"], as_index=False)["Win"].mean()
            .rename(columns={"Win": "AwayNeutWinPct"}))


def tourney_experience(tourney, years=EXP_YEARS):
    """Tournament games and wins by each team in the previous `years` seasons (strictly earlier)."""
    w = pd.DataFrame({"Season": tourney["Season"], "TeamID": tourney["WTeamID"], "G": 1, "W": 1})
    l = pd.DataFrame({"Season": tourney["Season"], "TeamID": tourney["LTeamID"], "G": 1, "W": 0})
    per = pd.concat([w, l]).groupby(["Season", "TeamID"], as_index=False)[["G", "W"]].sum()
    shifted = []
    for k in range(1, years + 1):
        p = per.copy()
        p["Season"] = p["Season"] + k          # results of season s count towards season s+k
        shifted.append(p)
    exp = pd.concat(shifted).groupby(["Season", "TeamID"], as_index=False)[["G", "W"]].sum()
    return exp.rename(columns={"G": "ExpGames", "W": "ExpWins"})


def build_team_feats(g):
    c = clean[g]
    keys = ["Season", "TeamID"]
    tf = season_win_pct(c["reg"])
    tf = tf.merge(c["seeds"], on=keys, how="left")
    tf = tf.merge(away_neutral_win_pct(c["reg"]), on=keys, how="left")
    tf = tf.merge(tourney_experience(c["tourney"]), on=keys, how="left")
    tf["AwayNeutWinPct"] = tf["AwayNeutWinPct"].fillna(tf["WinPct"])
    tf[["ExpGames", "ExpWins"]] = tf[["ExpGames", "ExpWins"]].fillna(0)
    return tf


def games_to_pairs(games):
    """2 rows per game (winner-as-A, loser-as-A). Margin is A's point margin; Result is 1/0."""
    m = games["WScore"] - games["LScore"]
    won = pd.DataFrame({"Season": games["Season"], "A": games["WTeamID"], "B": games["LTeamID"],
                        "Result": 1, "Margin": m})
    lost = pd.DataFrame({"Season": games["Season"], "A": games["LTeamID"], "B": games["WTeamID"],
                         "Result": 0, "Margin": -m})
    return pd.concat([won, lost], ignore_index=True)


def attach_features(pairs, team_feats, feature_cols):
    keep = ["Season", "TeamID"] + feature_cols
    fa = team_feats[keep].rename(columns={"TeamID": "A", **{c: f"{c}_A" for c in feature_cols}})
    fb = team_feats[keep].rename(columns={"TeamID": "B", **{c: f"{c}_B" for c in feature_cols}})
    out = pairs.merge(fa, on=["Season", "A"], how="left").merge(fb, on=["Season", "B"], how="left")
    for c in feature_cols:
        out[f"{c}_diff"] = out[f"{c}_A"] - out[f"{c}_B"]
    return out


def diff_cols(feature_cols):
    return [f"{c}_diff" for c in feature_cols]


BASE_FEATURES = ["SeedNum", "WinPct"]

team_feats, PAIRS = {}, {}
for g in GENDERS:
    team_feats[g] = build_team_feats(g)
    tour = clean[g]["tourney"]
    start = tour["Season"].min() + EXP_YEARS      # earlier seasons have an incomplete experience history
    usable = tour[(tour["Season"] >= start) & (tour["Season"] < PREDICT_SEASON)]
    PAIRS[g] = games_to_pairs(usable)
    print(f"{g}: {len(usable)} tournament games ({start}-{PREDICT_SEASON - 1}) -> {len(PAIRS[g])} pair rows")


In [ ]:
# Step 4 - Base Model

from scipy.stats import norm
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import brier_score_loss
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


def make_baseline_model():
    """Reference model from the shared baseline: plain logistic, no intercept."""
    return LogisticRegression(fit_intercept=False)


def make_scaled_logit():
    """Same model, features rescaled (no centring, so A/B symmetry is kept)."""
    return Pipeline([("scale", StandardScaler(with_mean=False)),
                     ("lr", LogisticRegression(fit_intercept=False))])


class MarginModel:
    """Predict A's point margin with ridge regression, then convert to P(A wins) = Phi(margin / sigma).
    target='Margin' tells the harness to fit on point margin instead of the win/loss label."""
    target = "Margin"

    def __init__(self, alpha=1.0):
        self.alpha = alpha

    def fit(self, X, y):
        X = np.asarray(X, float)
        self.s = np.sqrt((X ** 2).mean(axis=0))                 # RMS scale, no centring
        self.reg = Ridge(alpha=self.alpha, fit_intercept=False).fit(X / self.s, y)
        resid = np.asarray(y, float) - self.reg.predict(X / self.s)
        self.sigma = float(np.sqrt(np.mean(resid ** 2)))        # typical size of a margin error
        self.coef_ = (self.reg.coef_ / self.s).reshape(1, -1)   # in original units, for reporting
        return self

    def predict_proba(self, X):
        mu = self.reg.predict(np.asarray(X, float) / self.s)
        p = norm.cdf(mu / self.sigma)
        return np.column_stack([1 - p, p])


def make_margin_model():
    return MarginModel(alpha=1.0)


def evaluate_forward(pairs, feats, feature_cols, make_model):
    """Train on seasons < S, score on S. Brier is always computed against the win/loss Result."""
    data = attach_features(pairs, feats, feature_cols)
    X_cols = diff_cols(feature_cols)
    data = data.dropna(subset=X_cols)
    rows = []
    for season in sorted(data["Season"].unique()):
        if season < FIRST_VAL_SEASON:
            continue
        train = data[data["Season"] < season]
        valid = data[data["Season"] == season]
        model = make_model()
        target_col = getattr(model, "target", "Result")
        model.fit(train[X_cols], train[target_col])
        p = model.predict_proba(valid[X_cols])[:, 1]
        rows.append({"Season": season, "Games": len(valid) // 2,
                     "Brier": brier_score_loss(valid["Result"], p)})
    return pd.DataFrame(rows)


RESULTS, PER_SEASON = [], {}


def run_experiment(exp_id, feature_cols, make_model, change="", notes="", vs=None):
    per_season = {g: evaluate_forward(PAIRS[g], team_feats[g], feature_cols, make_model) for g in GENDERS}
    num = {g: (per_season[g]["Brier"] * per_season[g]["Games"]).sum() for g in GENDERS}
    den = {g: per_season[g]["Games"].sum() for g in GENDERS}
    men, women = num["M"] / den["M"], num["W"] / den["W"]
    pooled = (num["M"] + num["W"]) / (den["M"] + den["W"])        # game-weighted, like Kaggle
    ref = next((r for r in RESULTS if r["Exp"] == vs), RESULTS[0] if RESULTS else None)
    delta = pooled - ref["Pooled"] if ref else 0.0

    better = "n/a"                                                # in how many season-cells did we beat the reference?
    if ref and all(len(per_season[g]) == len(PER_SEASON[ref["Exp"]][g]) for g in GENDERS):
        wins = sum(int((per_season[g]["Brier"].values < PER_SEASON[ref["Exp"]][g]["Brier"].values).sum())
                   for g in GENDERS)
        better = f"{wins}/{sum(len(per_season[g]) for g in GENDERS)}"

    PER_SEASON[exp_id] = per_season
    RESULTS.append({"Exp": exp_id, "Features": "+".join(feature_cols),
                    "Model": getattr(make_model, "__name__", "model"),
                    "Men": round(men, 4), "Women": round(women, 4), "Pooled": round(pooled, 4),
                    "Games": int(den["M"] + den["W"]), "Vs": ref["Exp"] if ref else exp_id,
                    "Delta": round(delta, 4), "SeasonsBetter": better, "Change": change, "Notes": notes})
    pd.DataFrame(RESULTS).to_csv(f"{OUTPUT_DIR}/results.csv", index=False)
    print(f"{exp_id}: men {men:.4f} | women {women:.4f} | pooled {pooled:.4f} "
          f"| delta vs {RESULTS[-1]['Vs']}: {delta:+.4f} | seasons better: {better}")
    return per_season


run_experiment("E00_baseline", BASE_FEATURES, make_baseline_model,
               change="baseline", notes="seed diff + win% diff, plain logistic")
